# Phase 0 — EDA Gate: Healthcare Appointment No-Show

**Dataset:** `KaggleV2-May-2016.csv` (110,527 rows)

**Goal:** Validate 5 quality gates before proceeding to domain model pivot and training.

**Gates:**
1. Target rate 15-30%
2. Logistic AUC ≥ 0.65
3. No single feature >80% importance
4. No missing values
5. Sufficient volume (≥50k rows)

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit

%matplotlib inline
sns.set_style("whitegrid")

df = pd.read_csv("../data/raw/KaggleV2-May-2016.csv")
print(f"Rows: {len(df):,}  Cols: {df.shape[1]}")
df.head()

## 1. Schema & Data Quality

In [ ]:
schema = pd.DataFrame({
    "dtype": df.dtypes,
    "nulls": df.isnull().sum(),
    "null_pct": (df.isnull().sum() / len(df) * 100).round(2),
    "nunique": df.nunique(),
})
schema

## 2. Parse Dates & Create Target

In [ ]:
df["ScheduledDay"] = pd.to_datetime(df["ScheduledDay"])
df["AppointmentDay"] = pd.to_datetime(df["AppointmentDay"])
df["no_show"] = (df["No-show"] == "Yes").astype(int)
df["lead_time_days"] = (df["AppointmentDay"] - df["ScheduledDay"]).dt.days

neg_lead = (df["lead_time_days"] < 0).sum()
print(f"Negative lead_time rows: {neg_lead} ({neg_lead/len(df)*100:.2f}%) — clipping to 0")
df["lead_time_days"] = df["lead_time_days"].clip(lower=0)

target_rate = df["no_show"].mean()
print(f"\nNo-show rate: {target_rate*100:.1f}%")

## 3. Target by Segment

In [ ]:
df["age_band"] = pd.cut(df["Age"], bins=[-1, 18, 35, 55, 70, 120],
                        labels=["0-18", "19-35", "36-55", "56-70", "71+"])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

df.groupby("Gender")["no_show"].mean().plot.bar(ax=axes[0], title="No-show by Gender", rot=0)
df.groupby("age_band", observed=True)["no_show"].mean().plot.bar(ax=axes[1], title="No-show by Age Band", rot=0)
df.groupby("SMS_received")["no_show"].mean().plot.bar(ax=axes[2], title="No-show by SMS Received", rot=0)

for ax in axes:
    ax.axhline(target_rate, color="red", ls="--", alpha=0.7)
    ax.set_ylabel("No-show rate")

plt.tight_layout()
plt.show()

## 4. Lead Time vs No-Show

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

df[df["lead_time_days"] <= 60].groupby("lead_time_days")["no_show"].mean().plot(
    ax=ax[0], title="No-show rate by lead time (days)")
ax[0].set_ylabel("No-show rate")
ax[0].axhline(target_rate, color="red", ls="--", label=f"Overall {target_rate:.2f}")
ax[0].legend()

df["lead_time_days"].clip(upper=60).hist(bins=60, ax=ax[1])
ax[1].set_title("Lead time distribution (clipped 60d)")
ax[1].set_xlabel("Days")

plt.tight_layout()
plt.show()

print(df["lead_time_days"].describe())

## 5. SMS Confound Analysis

⚠️ **SMS_received=1 shows HIGHER no-show rate** — this is Simpson's paradox.
SMS is sent as intervention to patients with longer lead times (high-risk). Not a causal predictor.

In [ ]:
sms_tbl = df.groupby("SMS_received").agg(
    no_show_rate=("no_show", "mean"),
    mean_lead_time=("lead_time_days", "mean"),
    count=("no_show", "count"),
)
sms_tbl

## 6. Duplicate Patients

In [ ]:
patient_counts = df["PatientId"].value_counts()
print(f"Unique patients: {df['PatientId'].nunique():,}")
print(f"Patients with >1 appt: {(patient_counts > 1).sum():,} ({(patient_counts > 1).mean()*100:.1f}%)")
print(f"Max appts per patient: {patient_counts.max()}")
print("\n→ Must use GroupShuffleSplit to prevent leakage")

## 7. Correlation Heatmap

In [ ]:
numeric_cols = ["Age", "lead_time_days", "Scholarship", "Hipertension",
               "Diabetes", "Alcoholism", "Handcap", "SMS_received", "no_show"]
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(df[numeric_cols].corr(), annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax)
ax.set_title("Feature Correlations")
plt.tight_layout()
plt.show()

## 8. Logistic Baseline (Grouped Holdout)

In [ ]:
feature_cols = ["Age", "lead_time_days", "Scholarship", "Hipertension",
               "Diabetes", "Alcoholism", "Handcap", "SMS_received", "Gender_enc", "neigh_rank"]

df["Gender_enc"] = (df["Gender"] == "F").astype(int)
top20 = df["Neighbourhood"].value_counts().head(20).index
df["neigh_rank"] = df["Neighbourhood"].map({n: i for i, n in enumerate(top20)}).fillna(20).astype(int)

X = df[feature_cols].values
y = df["no_show"].values
groups = df["PatientId"].values

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X[train_idx], y[train_idx])
y_prob = lr.predict_proba(X[test_idx])[:, 1]
auc = roc_auc_score(y[test_idx], y_prob)

print(f"Logistic AUC (grouped holdout): {auc:.4f}")

coef_df = pd.DataFrame({
    "feature": feature_cols,
    "coef": lr.coef_[0],
    "abs_coef": np.abs(lr.coef_[0]),
}).sort_values("abs_coef", ascending=False)
coef_df["pct_importance"] = (coef_df["abs_coef"] / coef_df["abs_coef"].sum() * 100).round(1)

coef_df[["feature", "coef", "pct_importance"]]

## 9. Gate Verdicts

In [ ]:
max_pct = coef_df["pct_importance"].max()

gates = {
    "1. Target rate 15-30%": (0.15 <= target_rate <= 0.30, f"{target_rate*100:.1f}%"),
    "2. Logistic AUC >= 0.65": (auc >= 0.65, f"{auc:.4f}"),
    "3. No single feature >80%": (max_pct < 80, f"max {max_pct:.1f}%"),
    "4. No missing values (raw)": (df[["PatientId","AppointmentID","Gender","Age",
        "Neighbourhood","Scholarship","Hipertension","Diabetes","Alcoholism",
        "Handcap","SMS_received","No-show"]].isnull().sum().sum() == 0, "0 nulls"),
    "5. Sufficient volume (>=50k)": (len(df) >= 50000, f"{len(df):,} rows"),
}

all_pass = True
for name, (passed, val) in gates.items():
    status = "✅ PASS" if passed else "❌ FAIL"
    print(f"  {status}  {name}: {val}")
    if not passed:
        all_pass = False

verdict = "GO 🟢" if all_pass else "NO-GO 🔴"
print(f"\nOVERALL VERDICT: {verdict}")